# Getting Started with PID Theorem Prover

This notebook provides an interactive introduction to the PID Theorem Prover toolkit.

**What you'll learn:**
- How to validate PID measures
- Explore conflicts (no-go theorems)
- Discover implications between properties
- Infer mandatory and forbidden properties
- Visualize the property landscape

**Prerequisites:** Python 3.10+, Z3 solver, pandas

## Installation

If you haven't installed the package yet:

```bash
pip install -e .
```

In [ ]:
# Import the toolkit
from pid_prover import (
    validate_measures,
    measures_df,
    find_minimal_conflicts,
    find_minimal_implications,
    infer_property_consequences,
    find_maximal_compatible_sets,
    get_pid_solver
)

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

## 1. Exploring Available Measures

The toolkit includes 19 PID measures from the literature. Let's see what's available:

In [ ]:
# View the measures DataFrame
print(f"Total measures: {len(measures_df.columns)}")
print(f"Total properties: {len(measures_df.index)}")
print(f"\nMeasures: {list(measures_df.columns)}")
print(f"\nProperties: {list(measures_df.index)}")

In [ ]:
# Display the measures table
measures_df

### Encoding Legend

- **Y**: Property satisfied
- **Y_2**: Property satisfied only for bivariate (n=2)
- **Y_***: Property empirically supported
- **N**: Property not satisfied
- **N_A**: Property not applicable
- **N_dd**: Property fails for specific technical reasons

## 2. Validating Measures

The core functionality is checking whether a measure's claimed properties are logically consistent:

In [ ]:
# Validate all measures (this will take a minute)
validate_measures()

## 3. Finding Conflicts (No-Go Theorems)

Conflicts are minimal sets of properties that cannot coexist. These are "no-go theorems".

In [ ]:
# Find conflicts for bivariate case (n=2)
biv_conflicts = find_minimal_conflicts("bivariate", max_size=4)

print(f"\nFound {len(biv_conflicts)} minimal conflicts (bivariate, up to size 4)")
for i, conflict in enumerate(biv_conflicts, 1):
    print(f"{i}. {{{', '.join(conflict)}}}")

In [ ]:
# Find conflicts for multivariate case (n≥3)
multi_conflicts = find_minimal_conflicts("multivariate", max_size=5)

print(f"\nFound {len(multi_conflicts)} minimal conflicts (multivariate, up to size 5)")
for i, conflict in enumerate(multi_conflicts, 1):
    print(f"{i}. {{{', '.join(conflict)}}}")

**Key Finding:** The multivariate case has more conflicts than bivariate, showing it's more restrictive.

## 4. Finding Implications

Implications show which properties logically follow from others.

In [ ]:
# Find 2-premise implications for bivariate case
biv_implications = find_minimal_implications("bivariate", premise_size=2)

print(f"\nFound {len(biv_implications)} minimal implications (bivariate)")
for premises, conclusion in biv_implications[:10]:  # Show first 10
    premise_str = ' ∧ '.join(premises)
    print(f"{premise_str} ⇒ {conclusion}")

## 5. Property Inference

Given a set of desired properties, what additional properties are mandatory or forbidden?

In [ ]:
# Example: Design a measure with SR, S0, and GP
desired_properties = ['SR', 'S0', 'GP']

print(f"Desired properties: {{{', '.join(desired_properties)}}}")
print("\nFor bivariate (n=2):")

mandatory, forbidden = infer_property_consequences(desired_properties, "bivariate")

if mandatory:
    print(f"Mandatory: {{{', '.join(mandatory)}}}")
else:
    print("Mandatory: (none)")

if forbidden:
    print(f"Forbidden: {{{', '.join(forbidden)}}}")
else:
    print("Forbidden: (none)")

In [ ]:
# Same analysis for multivariate
print("\nFor multivariate (n≥3):")

mandatory, forbidden = infer_property_consequences(desired_properties, "multivariate")

if mandatory:
    print(f"Mandatory: {{{', '.join(mandatory)}}}")
else:
    print("Mandatory: (none)")

if forbidden:
    print(f"Forbidden: {{{', '.join(forbidden)}}}")
else:
    print("Forbidden: (none)")

## 6. Finding Maximal Compatible Sets

What are the largest consistent property sets?

In [ ]:
# Find maximal compatible sets (this may take a minute)
maximal_sets = find_maximal_compatible_sets("bivariate", min_size=10)

print(f"\nFound {len(maximal_sets)} maximal compatible sets (size ≥ 10)")
for i, mset in enumerate(maximal_sets[:5], 1):  # Show first 5
    print(f"{i}. Size {len(mset)}: {{{', '.join(sorted(mset))}}}")

## 7. Analyzing a Specific Measure

Let's analyze I_BROJA in detail:

In [ ]:
# Get I_BROJA properties
from pid_prover.measures.validation import dataframe_to_property_list

measure_name = "I_BROJA"

biv_props = dataframe_to_property_list(measure_name, "bivariate")
print(f"{measure_name} (bivariate): {{{', '.join(sorted(biv_props))}}}")
print(f"Total properties: {len(biv_props)}")

In [ ]:
# Check what's mandatory given I_BROJA's properties
mandatory, forbidden = infer_property_consequences(biv_props, "bivariate")

print(f"\nWith these {len(biv_props)} properties:")
print(f"Mandatory (implied): {mandatory if mandatory else '(none)'}")
print(f"Forbidden (incompatible): {forbidden if forbidden else '(none)'}")

## 8. Direct Z3 Solver Usage

For advanced users, you can use the Z3 solver directly:

In [ ]:
from pid_prover.solver.axioms import P
import z3

# Get solver with axioms
solver = get_pid_solver("bivariate")

# Check if {SR, GP, LP1} is consistent
test_props = ['SR', 'GP', 'LP1']
solver.push()
solver.add(z3.And([P[prop] for prop in test_props]))
result = solver.check()
solver.pop()

print(f"{{{', '.join(test_props)}}}: {result}")
print(f"{'Consistent' if result == z3.sat else 'Inconsistent'}")

## 9. Visualization: Property Heatmap

Visualize which measures have which properties:

In [ ]:
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    # Convert to binary matrix
    binary_df = measures_df.applymap(lambda x: 1 if x in ['Y', 'Y_2', 'Y_*'] else 0)
    
    plt.figure(figsize=(15, 10))
    sns.heatmap(binary_df, cmap='YlOrRd', cbar_kws={'label': 'Property Satisfied'},
                linewidths=0.5, linecolor='gray')
    plt.title('PID Measures Property Heatmap', fontsize=16)
    plt.xlabel('Measures', fontsize=12)
    plt.ylabel('Properties', fontsize=12)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Install matplotlib and seaborn for visualizations:")
    print("pip install matplotlib seaborn")

## 10. Summary

You now know how to:
- ✅ Validate PID measures
- ✅ Find conflicts and implications
- ✅ Infer property consequences
- ✅ Analyze measure compatibility
- ✅ Use the Z3 solver directly

**Next Steps:**
- See `02_reproducing_paper_results.ipynb` for paper reproduction
- See `03_measure_design_workshop.ipynb` for designing new measures
- Run exploration tests: `pytest tests/test_exploration_*.py`

## Exercise: Design Your Own Measure

Try designing a measure with specific properties and check consistency:

In [ ]:
# Your turn! Design a measure with these properties:
my_properties = ['SR', 'S0', 'M0', 'GP']  # Modify this

# Check consistency
solver = get_pid_solver("multivariate")
solver.push()
solver.add(z3.And([P[prop] for prop in my_properties]))
is_consistent = solver.check() == z3.sat
solver.pop()

print(f"Your measure: {{{', '.join(my_properties)}}}")
print(f"Consistent for n≥3: {is_consistent}")

if is_consistent:
    mandatory, forbidden = infer_property_consequences(my_properties, "multivariate")
    print(f"\nMandatory properties: {mandatory if mandatory else '(none)'}")
    print(f"Forbidden properties: {forbidden if forbidden else '(none)'}")
else:
    print("\n❌ This combination is inconsistent!")
    print("Try removing a property or using different ones.")